# CNN Architecture Implementation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [2]:
# load the dataset
transform = transforms.Compose(
    [
        transforms.ToTensor(), # converts images to pytorch tensors of shape (C, H, W) and normalizes pixel values to [0, 1]
        transforms.Normalize((0.5,), (0.5,))
    ]
)

In [3]:
train_dataset = datasets.CIFAR10(
    root="./data",train=True,download=True,transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",train=False,download=True,transform=transform)

train_dataset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [4]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,batch_size=64,shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,batch_size=64,shuffle=False)

In [5]:
# viewing the image dimension

images , labels = next(iter(train_loader))
images.shape[1:]
labels.unique().size()

torch.Size([10])

## cnn architecture to implement
```
Input (3×32×32)
↓
Conv(3*3, 3 → 16) + ReLU + MaxPool(2*2,s=2)
↓
Conv(3*3, 16 → 32) + ReLU + MaxPool(2*2, s=2)
↓
Flatten
↓
Fully Connected (Linear)
↓
Output (10 classes)
```

In [6]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(
            kernel_size=3,
            padding=1,
            in_channels=3,
            out_channels=16
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
        
        self.conv2 = nn.Conv2d(
            kernel_size=3,
            padding=1,
            in_channels=16,
            out_channels=32
        )

        self.fc1 = nn.Linear(32*8*8,128)
        self.fc2 = nn.Linear(128,10)

    def forward(self,x):

        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        # remember to flatten the dataset here
        x = x.view(x.size(0),-1)
        # what this function does
        """
        
        x.size() -> returns [B,C,H,W]
        x.size(0) -> returns B

        the view function transforms x to the given dimension , only the first dimension is explicitly mentioned
        -1 as 2nd argument tells pytorch to determine the rest by itself.
        
        """
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

## Same architecture of SimpleCNN using nn.Sequential
### When NOT to use nn.Sequential

You’ll stop using Sequential when:

- You build Residual blocks

- You build Inception modules

- You reuse tensors (x + residual)

- You need conditional logic

**That’s why ResNet switches back to explicit forward() logic.**

In [11]:
class SimpleCNNSequential(nn.Module):

    def __init__(self):

        super(SimpleCNNSequential,self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(
                kernel_size=3,
                padding=1,
                in_channels=3,
                out_channels=16
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),
            nn.Conv2d(
                kernel_size=3,
                padding=1,
                in_channels=16,
                out_channels=32
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),
            # fc neural network
            nn.Flatten(),
            nn.Linear(32*8*8,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )
    
    def forward(self,x):
        return self.model(x)

## CNN With Residual Connection

### High-level Architecture
```
Input (3 × 32 × 32)
↓
Stem Conv Block
↓
Residual Block A
↓
Residual Block B (spatial downsampling)
↓
Classifier
```

### Stem Conv Block
```
Conv2d(
  kernel_size = 3,
  stride = 1,
  padding = 1
)
ReLU
MaxPool2d(
  kernel_size = 2,
  stride = 2
)
```

### Residual Block A (Same Spatial Size)
```
Input: (B, C, H, W)

Main path:
  Conv2d(kernel_size=3, stride=1, padding=1)
  ReLU
  Conv2d(kernel_size=3, stride=1, padding=1)

Skip path:
  Identity (x)

Output:
  ReLU(main + skip)
  ```

### Residual Block B (Spatial Downsampling)

```
Input: (B, C, H, W)

Main path:
  Conv2d(kernel_size=3, stride=2, padding=1)
  ReLU
  Conv2d(kernel_size=3, stride=1, padding=1)

Skip path:
  Conv2d(kernel_size=3, stride=2, padding=1)

Output:
  ReLU(main + skip)
```

### Classifier
```
AdaptiveAvgPool2d((1,1))
Flatten
Linear → 10
```


In [35]:
# Reusable Residual Block
import torch.nn.functional as F

class ResidualBlock(nn.Module):

    def __init__(self,in_channels=16,out_channels=16,stride=1):
        super(ResidualBlock,self).__init__()

        self.conv1 = nn.Conv2d(
            kernel_size=3,
            stride=stride,
            padding=1,
            in_channels=in_channels,
            out_channels=out_channels
        )

        self.conv2 = nn.Conv2d(
            kernel_size=3,
            stride=1,
            padding=1,
            in_channels=out_channels,
            out_channels=out_channels
        )

        self.shortcut = nn.Sequential()

        if stride!=1 or in_channels!=out_channels:
            self.shortcut = nn.Conv2d(
                kernel_size=3,
                stride=stride,
                padding=1,
                in_channels=in_channels,
                out_channels=out_channels
            )
    
    def forward(self,x):

        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        out += self.shortcut(x)
        out = F.relu(out)

        return out

In [43]:
# Main Architecture
class SimpleCNNResidual(nn.Module):

    def __init__(self):
        super(SimpleCNNResidual,self).__init__()

        # stem block

        self.stem = nn.Sequential(
            nn.Conv2d(
                kernel_size=3,
                padding=1,
                in_channels=3,
                out_channels=16
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        # residual block A

        self.resA = ResidualBlock()
        
        # residual block B

        self.resB = ResidualBlock(
            in_channels=16,
            out_channels=32,
            stride=2
        )

        self.adaptivePool = nn.AdaptiveAvgPool2d((1,1))

        self.fc1 = nn.Linear(32,16)
        self.fc2 = nn.Linear(16,10)
    
    def forward(self,x):

        x = self.stem(x)
        x = self.resA(x)
        x = self.resB(x)
        x = self.adaptivePool(x)

        # flattening
        x = x.view(x.size(0),-1)
        x = self.fc1(x)

        return self.fc2(x)

## Debugging

In [44]:
# using dummy input
# model = SimpleCNN()
# model = SimpleCNNSequential()
model = SimpleCNNResidual()
dummy_input = torch.randn(1,3,32,32) # [B,C,H,W]
output = model(dummy_input)
output

tensor([[-0.1512,  0.0743,  0.1513,  0.0248,  0.0440, -0.2449,  0.2537,  0.0808,
          0.2159,  0.0114]], grad_fn=<AddmmBackward0>)

In [38]:
# Using torchinfo
from torchinfo import summary
summary(model,input_size=(1,3,32,32))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleCNNResidual                        [1, 10]                   --
├─Sequential: 1-1                        [1, 16, 16, 16]           --
│    └─Conv2d: 2-1                       [1, 16, 32, 32]           448
│    └─ReLU: 2-2                         [1, 16, 32, 32]           --
│    └─MaxPool2d: 2-3                    [1, 16, 16, 16]           --
├─ResidualBlock: 1-2                     [1, 16, 16, 16]           --
│    └─Conv2d: 2-4                       [1, 16, 16, 16]           2,320
│    └─Conv2d: 2-5                       [1, 16, 16, 16]           2,320
│    └─Sequential: 2-6                   [1, 16, 16, 16]           --
├─ResidualBlock: 1-3                     [1, 32, 8, 8]             --
│    └─Conv2d: 2-7                       [1, 32, 8, 8]             4,640
│    └─Conv2d: 2-8                       [1, 32, 8, 8]             9,248
│    └─Conv2d: 2-9                       [1, 32, 8, 8]             4,640

## NiN Block
```
k x k conv
relu
1 x 1 conv
relu
1 x 1 conv
relu 
```

In [1]:
class NiNBlock(nn.Module):

    def __init__(self,in_channels,out_channels,kernel_size,padding,stride):
        super(NiNBlock,self).__init__()

        self.nin = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                padding=padding,
                stride = stride
            ),
            nn.ReLU(),
            nn.Conv2d(
                kernel_size=1,
                in_channels=out_channels,
                out_channels=out_channels
            ),
            nn.ReLU(),
            nn.Conv2d(
                kernel_size=1,
                in_channels=out_channels,
                out_channels=out_channels
            ),
            nn.ReLU()
        )
    
    def forward(self,x):

        return self.nin(x)


NameError: name 'nn' is not defined

# training and evaluation
Forward → Loss → Backward → Update


In [40]:
def train_model(model,num_epochs=5):

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(),lr=0.001)

    for i in range(num_epochs):

        running_loss = 0.0

        for images,labels in train_loader:
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output,labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"epoch [{i+1}/{num_epochs}] loss: {running_loss:.4f}")

## Evaluating the model

In [8]:
def evaluate_model(model):

    correct = 0
    total = 0

    # switch to eval mode

    model.eval()

    with torch.no_grad():

        for images,labels in test_loader:

            outputs = model(images)

            _,predicted = torch.max(outputs,1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
             

In [45]:
train_model(model)

epoch [1/5] loss: 1416.4967
epoch [2/5] loss: 1229.6927
epoch [3/5] loss: 1151.8889
epoch [4/5] loss: 1088.7346
epoch [5/5] loss: 1025.5459


In [46]:
evaluate_model(model)

Test Accuracy: 53.97%
